# 3. Multicollinearity Analysis
Variance Inflation Factor (VIF), Pearson correlation, and Spearman correlation analysis for the 8 flood conditioning factors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

## Load Data / 

In [ ]:
# Pathlib Path for output directory
PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "samples"

# Sample training data (output from Notebook 1)
SAMPLE_TRAINING = OUTPUT_DIR / 'dataset_training.csv'

df = pd.read_csv(SAMPLE_TRAINING)

FACTORS = ['elevation', 'slope', 'distance_to_river', 'distance_to_coast',
    'land_cover', 'soil_type', 'ndvi', 'rainfall']

X = df[FACTORS]
print(f'Dataset: {len(df)} samples ({len(df[df.label==1])} flood, {len(df[df.label==0])} non-flood)')

## 1. Variance Inflation Factor (VIF) / 

Threshold: VIF < 5 = OK | VIF 5-10 = moderate | VIF > 10 = severe

In [ ]:
def calculate_vif(X):
    """Calculate VIF for each feature using sklearn LinearRegression."""
    vif_values = []
    for i in range(X.shape[1]):
        X_other = np.delete(X.values, i, axis=1)
        y = X.values[:, i]
        reg = LinearRegression().fit(X_other, y)
        r_squared = reg.score(X_other, y)
        vif = 1 / (1 - r_squared) if r_squared < 1 else float('inf')
        vif_values.append(vif)
    return vif_values

vif_values = calculate_vif(X)
tolerance_values = [1/v if v != float('inf') else 0 for v in vif_values]

vif_df = pd.DataFrame({
    'Factor': FACTORS,
    'VIF': [round(v, 3) for v in vif_values],
    'Tolerance': [round(t, 4) for t in tolerance_values],
    'Status': ['OK' if v < 5 else ('MODERATE' if v < 10 else 'HIGH') for v in vif_values]
})
print('Variance Inflation Factor (VIF) Results:')
print(vif_df.to_string(index=False))

## 2. Pearson Correlation Matrix / 

In [ ]:
pearson_corr = X.corr(method='pearson')

plt.figure(figsize=(8, 7))
sns.heatmap(pearson_corr, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Pearson Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nHighly correlated pairs (|r| > 0.7):')
for i in range(len(FACTORS)):
    for j in range(i+1, len(FACTORS)):
        r = pearson_corr.iloc[i, j]
        if abs(r) > 0.7:
            print(f' {FACTORS[i]:<20} x {FACTORS[j]:<20} : r = {r:+.4f}')

## 3. Spearman Correlation Matrix / 

In [ ]:
spearman_corr = X.corr(method='spearman')

plt.figure(figsize=(8, 7))
sns.heatmap(spearman_corr, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Spearman Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nHighly correlated pairs (|rho| > 0.7):')
for i in range(len(FACTORS)):
    for j in range(i+1, len(FACTORS)):
        r = spearman_corr.iloc[i, j]
        if abs(r) > 0.7:
            print(f' {FACTORS[i]:<20} x {FACTORS[j]:<20} : rho = {r:+.4f}')

## 4. Summary & Conclusion / 

In [ ]:
max_vif_idx = np.argmax(vif_values)
max_vif = vif_values[max_vif_idx]
max_vif_factor = FACTORS[max_vif_idx]

print(f'Highest VIF: {max_vif:.3f} ({max_vif_factor})')
print(f'Lowest Tolerance: {tolerance_values[np.argmin(tolerance_values)]:.4f}')

if max_vif < 5:
    print('All VIF values < 5. No significant multicollinearity detected.')
    print('All 8 conditioning factors are retained for ML modeling.')
elif max_vif < 10:
    print(f'VIF 5-10 for: {max_vif_factor}. Moderate multicollinearity.')
else:
    print(f'VIF > 10 for: {max_vif_factor}. Severe multicollinearity.')

## 5. Descriptive Statistics / 

In [ ]:
X.describe().round(3)

## 6. Save Correlation Matrices / 

In [ ]:
pearson_corr.to_csv(PROJECT_ROOT / 'outputs/pearson_correlation.csv')
spearman_corr.to_csv(PROJECT_ROOT / 'outputs/spearman_correlation.csv')
vif_df.to_csv(PROJECT_ROOT / 'outputs/vif_results.csv', index=False)
print('Saved: outputs/pearson_correlation.csv, spearman_correlation.csv, vif_results.csv')